# Eagle3 Draft Model Training (Offline Mode)

Train Eagle3 draft model using pre-generated hidden states.

**Prerequisites:** Run all previous notebooks first!

**Estimated time:** 12-20 hours for 100K samples (A100 40GB)
**GPU memory:** ~10GB (much less than online mode!)
**Output:** Trained draft model for speculative decoding

## ✅ Advantages of Offline Mode

- **Low GPU Memory:** Only 10GB (vs 80GB for online mode)
- **Faster Training:** 12-20h (vs 25-40h)
- **Flexible:** Can restart training without regenerating data
- **Cheap Experiments:** Try different hyperparameters easily

## Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change to AngelSlim directory
%cd /content/AngelSlim

In [ ]:
# Install dependencies
!pip install -q transformers>=4.37.0 accelerate torch datasets deepspeed wandb

## Wandb Setup (Optional but Recommended)

In [ ]:
from google.colab import userdata
import wandb

# Get API key from Colab secrets
# Add your Wandb API key in Colab: Secrets tab -> WANDB_API_KEY
try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("✅ Wandb logged in successfully")
except Exception as e:
    print(f"⚠️ Wandb login failed: {e}")
    print("Training will continue without Wandb logging")

## Configuration

In [ ]:
CONFIG = {
    # Model configuration
    'target_model_name': 'Qwen/Qwen3-VL-30B-A3B',  # For loading LM head only
    # Для 4B модели: 'Qwen/Qwen3-VL-4B'
    
    'draft_config': 'angelslim/compressor/speculative/train/configs/qwen3-vl-30b-a3b-eagle3-mrope.json',
    # Для 4B: 'angelslim/compressor/speculative/train/configs/qwen3-vl-4b-eagle3-mrope.json'
    
    # Data paths
    'train_hidden_path': '/content/drive/MyDrive/Eagle3_Qwen3VL/hidden_states/train',
    'train_data_path': None,  # Not used in offline mode
    
    # Output
    'output_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/checkpoints/qwen3-vl-30b-eagle3',
    
    # Training hyperparameters
    'num_train_epochs': 3,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,  # Effective batch size = 1*8 = 8
    'learning_rate': 1e-4,
    'warmup_ratio': 0.1,
    'lr_scheduler_type': 'constant',
    
    # Checkpointing
    'save_strategy': 'steps',
    'save_steps': 500,
    'save_total_limit': 3,  # Keep only 3 most recent checkpoints
    
    # Logging
    'logging_steps': 20,
    'report_to': 'wandb',  # or 'none' if wandb not available
    'run_name': 'qwen3-vl-30b-eagle3-offline',
    
    # Other settings
    'model_max_length': 4096,
    'chat_template_type': 'qwen3_vl',
    'embed_weight_key': 'model.language_model.embed_tokens.weight',
    'lm_head_key': 'model.language_model.lm_head.weight',
    'num_proc': 8,
    'training_time_test_length': 7,  # Eagle3 prediction depth
    
    # DeepSpeed
    'deepspeed_config': 'angelslim/compressor/speculative/train/configs/deepspeed_zero2_colab.json',
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Quick Test with 2B Model (Optional)

Before running the full training, you can test with Qwen3-VL-2B on a small subset.

In [ ]:
# Uncomment to run quick test
# QUICK_TEST = True
QUICK_TEST = False

if QUICK_TEST:
    print("🧪 Running quick test with Qwen3-VL-2B")
    CONFIG['target_model_name'] = 'Qwen/Qwen3-VL-2B'
    CONFIG['draft_config'] = 'angelslim/compressor/speculative/train/configs/qwen3-vl-2b-eagle3-mrope.json'
    CONFIG['num_train_epochs'] = 1
    CONFIG['save_steps'] = 100
    CONFIG['run_name'] = 'qwen3-vl-2b-eagle3-test'
    print("✅ Test mode enabled")
else:
    print("🚀 Full training mode")

## Cache Target Model (for LM Head)

In [ ]:
from colab_code.utils import get_cache_manager

cache_manager = get_cache_manager()
model_cache_name = CONFIG['target_model_name'].replace('/', '_')

# Check if already cached
target_model_path = cache_manager.load_from_cache(model_cache_name)

if target_model_path is None:
    print(f"Downloading {CONFIG['target_model_name']}...")
    target_model_path = cache_manager.download_and_cache_model(
        model_name_or_path=CONFIG['target_model_name'],
        cache_name=model_cache_name,
    )
else:
    print(f"✅ Model already cached: {target_model_path}")

CONFIG['target_model_path'] = target_model_path

## Start Training

In [ ]:
import subprocess
import sys

print("="*60)
print("Starting Eagle3 Offline Training")
print("="*60)

# Build command
cmd = [
    sys.executable, 'tools/train_eagle3_offline.py',
    '--modal_type', 'VLM',
    '--target_model_name_or_path', CONFIG['target_model_path'],
    '--draft_model_config_path', CONFIG['draft_config'],
    '--train_hidden_path', CONFIG['train_hidden_path'],
    '--output_dir', CONFIG['output_dir'],
    '--num_train_epochs', str(CONFIG['num_train_epochs']),
    '--per_device_train_batch_size', str(CONFIG['per_device_train_batch_size']),
    '--gradient_accumulation_steps', str(CONFIG['gradient_accumulation_steps']),
    '--learning_rate', str(CONFIG['learning_rate']),
    '--warmup_ratio', str(CONFIG['warmup_ratio']),
    '--lr_scheduler_type', CONFIG['lr_scheduler_type'],
    '--save_strategy', CONFIG['save_strategy'],
    '--save_steps', str(CONFIG['save_steps']),
    '--save_total_limit', str(CONFIG['save_total_limit']),
    '--logging_steps', str(CONFIG['logging_steps']),
    '--model_max_length', str(CONFIG['model_max_length']),
    '--lm_head_key', CONFIG['lm_head_key'],
    '--embed_weight_key', CONFIG['embed_weight_key'],
    '--chat_template_type', CONFIG['chat_template_type'],
    '--deepspeed', CONFIG['deepspeed_config'],
    '--report_to', CONFIG['report_to'],
    '--run_name', CONFIG['run_name'],
    '--num_proc', str(CONFIG['num_proc']),
    '--training_time_test_length', str(CONFIG['training_time_test_length']),
    '--bf16',
]

# Add train_data_path only if it exists (offline mode might not need it)
if CONFIG['train_data_path']:
    cmd.extend(['--train_data_path', CONFIG['train_data_path']])

print("\nTraining command:")
print(' '.join(cmd))
print()

# Run training
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    print(f"\n❌ Training failed with code {result.returncode}")

## Training Progress (Run this in a separate cell while training)

You can monitor training progress by checking the logs or Wandb dashboard.

In [ ]:
# Check latest checkpoint
from pathlib import Path

output_dir = Path(CONFIG['output_dir'])
checkpoints = sorted(output_dir.glob('checkpoint-*'), 
                     key=lambda x: int(x.name.split('-')[1]) if x.name.split('-')[1].isdigit() else 0)

if checkpoints:
    latest = checkpoints[-1]
    print(f"Latest checkpoint: {latest.name}")
    
    # Try to read training state
    trainer_state = latest / 'trainer_state.json'
    if trainer_state.exists():
        import json
        with open(trainer_state) as f:
            state = json.load(f)
        
        if 'log_history' in state and state['log_history']:
            latest_log = state['log_history'][-1]
            print("\nLatest metrics:")
            for key, value in latest_log.items():
                if key.startswith('train/'):
                    print(f"  {key}: {value:.4f}")
else:
    print("No checkpoints found yet. Training may still be starting...")

## Validation and Summary

In [ ]:
import json
from pathlib import Path

output_dir = Path(CONFIG['output_dir'])
checkpoints = sorted(output_dir.glob('checkpoint-*'),
                     key=lambda x: int(x.name.split('-')[1]) if x.name.split('-')[1].isdigit() else 0)

print("="*60)
print("TRAINING SUMMARY")
print("="*60)

if checkpoints:
    print(f"\n📊 Checkpoints: {len(checkpoints)}")
    for ckpt in checkpoints:
        size_mb = sum(f.stat().st_size for f in ckpt.rglob('*') if f.is_file()) / (1024**2)
        print(f"  {ckpt.name}: {size_mb:.1f} MB")
    
    # Read final metrics
    latest = checkpoints[-1]
    trainer_state = latest / 'trainer_state.json'
    
    if trainer_state.exists():
        with open(trainer_state) as f:
            state = json.load(f)
        
        if 'log_history' in state:
            # Get final metrics
            final_metrics = {}
            for log_entry in reversed(state['log_history']):
                for key in ['train/loss', 'train/acc_0', 'train/acc_1']:
                    if key in log_entry and key not in final_metrics:
                        final_metrics[key] = log_entry[key]
                if len(final_metrics) >= 3:
                    break
            
            print("\n📈 Final Metrics:")
            for key, value in final_metrics.items():
                print(f"  {key}: {value:.4f}")
            
            # Success criteria
            print("\n✅ Success Criteria Check:")
            if 'train/loss' in final_metrics:
                if final_metrics['train/loss'] < 1.2:
                    print(f"  ✅ Loss < 1.2: {final_metrics['train/loss']:.4f}")
                else:
                    print(f"  ⚠️ Loss >= 1.2: {final_metrics['train/loss']:.4f}")
            
            if 'train/acc_0' in final_metrics:
                if final_metrics['train/acc_0'] > 0.6:
                    print(f"  ✅ Acceptance rate > 60%: {final_metrics['train/acc_0']*100:.1f}%")
                else:
                    print(f"  ⚠️ Acceptance rate <= 60%: {final_metrics['train/acc_0']*100:.1f}%")
    
    print(f"\n📁 Model saved to:")
    print(f"  {output_dir}")
    
    print(f"\n🎉 Training complete!")
    print(f"\nNext steps:")
    print(f"  1. Test the draft model with inference")
    print(f"  2. Measure speedup vs baseline")
    print(f"  3. Deploy for production use")
    
else:
    print("\n⚠️ No checkpoints found. Training may have failed.")
    print("Check the logs above for errors.")

## Storage Summary

In [ ]:
from colab_code.utils import get_cache_manager

cache_manager = get_cache_manager()
storage = cache_manager.estimate_storage(
    include_models=True,
    include_datasets=True,
    include_checkpoints=True,
    include_hidden_states=True,
)

print("\n💾 Google Drive Usage:")
print(f"  Models: {storage['models']:.2f} GB")
print(f"  Datasets: {storage['datasets']:.2f} GB")
print(f"  Hidden States: {storage['hidden_states']:.2f} GB")
print(f"  Checkpoints: {storage['checkpoints']:.2f} GB")
print(f"  Total: {storage['total']:.2f} GB / 1000 GB")
print(f"  Remaining: {1000 - storage['total']:.2f} GB")